In [ ]:
# =============================================================================
# Standard Library
# =============================================================================
import os                              # File paths and working-directory control
import math                            # Basic mathematical functions
import json                            # Read and write JSON files
import itertools                       # Cartesian products for experimental designs
from itertools import product as iterproduct
import warnings                        # Warning-message control
from functools import partialmethod    # Modify tqdm default behavior
from typing import Tuple, List         # Type hints


# =============================================================================
# Core Computing
# =============================================================================
import numpy as np                     # Array operations and gridded data
import pandas as pd                    # Tabular data handling
from scipy import stats                # Statistical summaries and tests
import scipy.signal as signal          # Moving-window and signal operations
from scipy.interpolate import RegularGridInterpolator  # Interpolate gridded fields
from scipy.ndimage import gaussian_filter              # Gaussian smoothing
from astropy.convolution import convolve               # General convolution


# =============================================================================
# Plotting
# =============================================================================
import matplotlib.pyplot as plt        # Main plotting interface
import matplotlib.gridspec as gridspec # Custom subplot layouts
import seaborn as sns                  # Statistical plotting
from matplotlib.colors import Normalize, TwoSlopeNorm  # Color normalization
from matplotlib.patches import Patch   # Custom legend patches
from matplotlib.ticker import MultipleLocator, AutoMinorLocator  # Axis tick control
from mpl_toolkits.axes_grid1 import make_axes_locatable          # Attach colorbars

plt.rc("axes", axisbelow=True)         # Draw grid lines below plotted data
cmap = plt.cm.viridis                  # Default colormap


# =============================================================================
# Progress Bars and Logging
# =============================================================================
from tqdm.auto import tqdm             # Notebook/script-compatible progress bars

tqdm.__init__ = partialmethod(
    tqdm.__init__,
    disable=True,
)                                      # Disable tqdm progress bars globally


# =============================================================================
# Machine Learning and Optimization
# =============================================================================
import optuna                          # Hyperparameter optimization
from sklearn.gaussian_process import GaussianProcessRegressor  # Gaussian-process model
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    RBF,
    WhiteKernel,
)                                      # Gaussian-process kernels
from sklearn.metrics import mean_squared_error  # Model error metric
from sklearn.preprocessing import StandardScaler # Feature standardization

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)                                      # Suppress Optuna trial-level logs


# =============================================================================
# Geostatistics
# =============================================================================
import geostatspy                      # Geostatistical workflows
import geostatspy.GSLIB as GSLIB       # GSLIB utilities and plotting wrappers
import geostatspy.geostats as geostats # GSLIB-style geostatistical methods


# =============================================================================
# Warning Control
# =============================================================================
ignore_warnings = True                 # Set to False to show warnings

if ignore_warnings:
    warnings.filterwarnings("ignore")

### **Grid Configuration**

In [ ]:
# --- Grid Parameters ---
rng = np.random.default_rng(44)
L = 1000.0           # domain size in meters (both x and y)
dx = 10             # grid spacing in meters

# --- Grid ---
nx = int(L / dx)
ny = int(L / dx)

x_cordinates = np.arange(0.0, L, dx)
y_cordinates = np.arange(0.0, L, dx)

### **Geostats Related Helper Functions**

In [ ]:
def scov(A, B):
    """Sample spatial covariance between two 2D arrays."""
    a = (A - A.mean()).ravel()
    b = (B - B.mean()).ravel()
    return float((a * b).mean())

def var1(x):
    x = np.asarray(x)
    return float(np.var(x, ddof=1))

def cov1(x,y):
    x = np.asarray(x) - np.mean(x)
    y = np.asarray(y) - np.mean(y)
    return float(np.dot(x,y) / (len(x)-1))
    
# variogram models
def variogram_model(h, model, range_, sill, nugget):
    """Create an analytical variogram model given its parameters."""
    c = sill - nugget
    hr = np.asarray(h, float) / range_
    if model.lower().startswith("exp"):
        gamma = nugget + c * (1.0 - np.exp(-3*hr))
    elif model.lower().startswith("gau"):
        gamma = nugget + c * (1.0 - np.exp(-3*(hr ** 2)))
    elif model.lower().startswith("sph"):
        gamma = np.where(
            hr <= 1.0,
            nugget + c * (1.5 * hr - 0.5 * hr**3),
            nugget + c,
        )
    else:
        raise ValueError("Model must be exponential, gaussian, or spherical")
    return gamma

#calculating covariance from variogram
def covariance_from_variogram(h, model, range_, sill, nugget):
    """Calculate covariance from specific analytical variogram model."""
    gamma = variogram_model(h, model, range_, sill, nugget)
    return (sill - nugget) - (gamma - nugget)

In [ ]:
# =============================================================================
# Core Function: Gram-Schmidt
# =============================================================================
# ====== Gram-Schmidt Helpers ======
def demean(A):
    """Return A with spatial mean removed."""
    return A - float(np.mean(A))

def rescale_to_var(A, v_target, eps=1e-12):
    """Center and scale A to have sample variance v_target."""
    A0 = A - A.mean()
    v  = float(A0.var())
    if v < eps:
        return A0  # nothing to scale (or degenerate); returns zero-mean A
    return A0 * np.sqrt(max(v_target, 0.0) / v)

def norm2(v, eps=1e-12):
    """Euclidean norm with floor to avoid division by zero."""
    return float(np.sqrt(np.dot(v, v))) + eps
    
def gram_schmidt_orthogonalization(A, B, eps=1e-12):
    """
    Make B orthogonal to A via Gram-Schmidt, using A as the reference direction.
    ~B is Trend, A is Residual

    Concept:
        Let a = demean(A), b = demean(B) (flattened).
        Remove the projection of b onto a:
            b_perp = b - alpha * a
        where:
            alpha = <b,a> / <a,a>

    This guarantees:
        <a, b_perp> = 0   (up to numerical precision)
    """
    if A.shape != B.shape:
        raise ValueError(f"A and B must have the same shape, got {A.shape} vs {B.shape}")

    if A.ndim not in (1, 2):
        raise ValueError(f"Expected A,B to be 1D or 2D arrays; got ndim={A.ndim}")
    Z = A+B
    # Work in zero-mean space
    A0 = demean(A)
    B0 = demean(B)

    # Flatten for inner products
    a = A0.ravel()
    b = B0.ravel()

    aa = float(np.dot(a, a))
    if aa < eps:
        # Degenerate reference: cannot define a direction
        # In this case, "orthogonalizing" is meaningless; return demeaned inputs.
        A_orth = A0.copy()
        B_orth = B0.copy()
        alpha = 0.0
        print("Degenerate reference: cannot define a direction")
    else:
        alpha = float(np.dot(b, a)) / (aa + eps)
        b_perp = b - alpha * a
        print(f"Projection Coeff: {alpha}")
        A_orth = A0.copy()
        B_orth = b_perp.reshape(B.shape)
    B_orth = B_orth + B.mean()
    return A_orth, B_orth



In [ ]:
def smooth_krige_GS_adjustments(sampled_df, dx, nx, ny):
    """Propagate Gram-Schmidt correction from well locations to the full grid using kriging."""
    
    # Range of adjustment values at well locations used to bound kriging estimates
    diff_min = sampled_df['GS_adjustments'].min()
    diff_max = sampled_df['GS_adjustments'].max()

    # Long-range variogram parameters captures the smooth, large-scale 
    # spatial structure of the correction field
    diff_vrange_maj = 800   # major range (m): long range to ensure smooth propagation
    diff_vrange_min = 800   # minor range (m): isotropic correction assumed
    diff_vazi       = 0.0   # variogram azimuth (degrees): no preferred direction
    diff_vrel_nugget = 0.2  # nugget effect: small nugget reflects spatial continuity of the correction field

    # Kriging mean and sill estimated from the sample adjustments
    diff_skmean = np.average(sampled_df['GS_adjustments'].values)  # global mean of corrections
    diff_sill   = np.var(sampled_df['GS_adjustments'].values)      # variance of corrections at wells

    # Build variogram model for the adjustment field
    # Exponential model with long range ensures smooth spatial interpolation
    por_vario = GSLIB.make_variogram(
        nug=diff_vrel_nugget, nst=1, it1=2,
        cc1=1.0 - diff_vrel_nugget,
        azi1=diff_vazi,
        hmaj1=diff_vrange_maj,
        hmin1=diff_vrange_min
    )

    # Kriging search and estimation parameters
    ktype  = 1   # kriging type: 1 = ordinary kriging (estimates local mean from neighbors)
    radius = 800 # search radius (m): includes all wells within range of each grid node
    nxdis  = 1   # grid discretizations for block kriging (1 = point kriging)
    nydis  = 1
    ndmin  = 5   # minimum number of neighboring wells required for an estimate
    ndmax  = 20  # maximum number of neighboring wells used per estimate

    # Cell-centered grid origin  ensures estimates align with simulation grid
    xmn = dx / 2.0
    ymn = dx / 2.0
    xsiz = dx
    ysiz = dx

    # Run 2D kriging to propagate the GS adjustment to all grid nodes
    # diff_kmap: kriged adjustment surface | diff_vmap: kriging variance surface
    diff_kmap, diff_vmap = geostats.kb2d(
        sampled_df, 'X', 'Y',
        'GS_adjustments',
        diff_min, diff_max,
        nx, xmn, xsiz,
        ny, ymn, ysiz,
        nxdis, nydis,
        ndmin, ndmax,
        radius, ktype,
        diff_skmean, por_vario
    )

    return diff_kmap  

### **Visualization Helper Function**

In [ ]:
def add_grid():
    plt.gca().grid(True, which='major',linewidth = 1.0); plt.gca().grid(True, which='minor',linewidth = 0.2) # add y grids
    plt.gca().tick_params(which='major',length=7); plt.gca().tick_params(which='minor', length=4)
    plt.gca().xaxis.set_minor_locator(AutoMinorLocator()); plt.gca().yaxis.set_minor_locator(AutoMinorLocator()) # turn on minor ticks   

### **Model Training Helper Functions**

In [ ]:
def compute_metrics(model, samples, Z_samples, true_trend_values, scaler=None):
    """
    Compute GP trend metrics.  If a fitted StandardScaler is supplied the GP
    prediction (which is in scaled space) is inverse-transformed back to the
    original porosity units before any metric is computed.
    """
    Z_samples = Z_samples.values if hasattr(Z_samples, 'values') else np.array(Z_samples)
    point_trend_raw = model.predict(samples)

    # Inverse-transform if the model was trained on scaled targets
    if scaler is not None:
        point_trend = scaler.inverse_transform(point_trend_raw.reshape(-1, 1)).ravel()
    else:
        point_trend = point_trend_raw

    point_residual = Z_samples - point_trend

    var_trend = var1(point_trend)
    var_resid = var1(point_residual)
    var_z     = var1(Z_samples)

    cov_attribution_pct = np.abs(var_z - (var_trend + var_resid)) / var_z
    var_resid_pct       = var_resid / var_z

    trend_rmse      = np.sqrt(mean_squared_error(true_trend_values, point_trend))
    trend_range     = np.ptp(true_trend_values)
    normalized_rmse = trend_rmse / trend_range if trend_range > 0 else trend_rmse

    return cov_attribution_pct, var_resid_pct, normalized_rmse, point_trend, point_residual

def var1(x):
    return np.var(x, ddof=1)


# =============================================================================
# GP Trend Estimation with Spatial Block CV
# =============================================================================

def spatial_block_folds(X, n_blocks_side=2):
    """
    Divide the spatial domain into n_blocks_side x n_blocks_side grid cells.
    Each fold leaves out one block so validation wells are geographically
    isolated from training wells.
    """
    x_edges = np.linspace(X[:, 0].min(), X[:, 0].max(), n_blocks_side + 1)
    y_edges = np.linspace(X[:, 1].min(), X[:, 1].max(), n_blocks_side + 1)

    # Use interior edges so np.digitize correctly assigns boundary wells
    xi = np.digitize(X[:, 0], x_edges[1:-1])   # 0 .. n_blocks_side-1
    yi = np.digitize(X[:, 1], y_edges[1:-1])
    block_id = xi * n_blocks_side + yi

    folds = []
    for b in range(n_blocks_side ** 2):
        val_idx   = np.where(block_id == b)[0]
        train_idx = np.where(block_id != b)[0]
        if len(val_idx) > 0 and len(train_idx) > 0:
            folds.append((train_idx, val_idx))

    print(f"  Spatial CV: {len(folds)} blocks, "
          f"avg {np.mean([len(v) for _, v in folds]):.0f} wells per validation block")
    return folds


# =========================================================================
# Kernel builder  signal variance FIXED, length_scale bounded
# =========================================================================

def _build_kernel(kernel_type, length_scale, noise_level, signal_variance,
                  nu=1.5, ls_min=325, ls_max=900):
    """
    Construct kernel with EXPLICIT length_scale_bounds so sklearn's internal
    MLL optimizer cannot escape the intended range.
    Full kernel: ÏƒÂ² * Matern(â„“) + Ïƒ_nÂ² * I
    ConstantKernel tunes ÏƒÂ² (signal amplitude)  separate from length scale.
    """
    ls_bounds   = (ls_min, ls_max)
    noise_floor = (1 - signal_variance) * 0.3
    noise_ceil  = (1 - signal_variance) * 1.5

    amplitude = ConstantKernel(
        constant_value        = signal_variance,
        constant_value_bounds = "fixed",
    )
    white_kernel = WhiteKernel(
        noise_level        = np.clip(noise_level, noise_floor, noise_ceil),
        noise_level_bounds = (noise_floor, noise_ceil)
    )
    if kernel_type == "rbf":
        return amplitude * RBF(length_scale=length_scale,
                               length_scale_bounds=ls_bounds) + white_kernel
    else:
        return amplitude * Matern(length_scale=length_scale,
                                  length_scale_bounds=ls_bounds,
                                  nu=nu) + white_kernel


In [ ]:


# =========================================================================
# Optuna objective - SINGLE objective (CV-MSE only)
# =========================================================================
def make_constrained_objective(samples, Z_samples, target_var_ratio,
                                n_blocks_side=2, ls_min=200, ls_max=900):
    """
    Spatial block-CV objective with an asymmetric variance-ratio penalty.

    Objective = mean_fold_MSE + lambda * max(ratio_gap, 0)^2

    where:
        ratio_gap = Var(predicted_trend_train) / Var(Z_train) - target_var_ratio

    The penalty is zero when the GP under-explains (ratio_gap <= 0) and
    quadratic when it over-explains, directly coupling the search to the
    variance budget rather than relying on kernel parameter constraints alone.

    The penalty is multiplicative (scaled by mean_fold_MSE) so it remains
    dominant regardless of porosity units or data scale.
    """
    X     = samples.values if hasattr(samples, 'values') else np.array(samples)
    y     = Z_samples.values if hasattr(Z_samples, 'values') else np.array(Z_samples)
    folds = spatial_block_folds(X, n_blocks_side=n_blocks_side)

    def objective(trial):
        length_scale = trial.suggest_float("length_scale", ls_min, ls_max, log=False)
        kernel_type  = trial.suggest_categorical("kernel_type", ["rbf", "matern"])
        nu           = trial.suggest_categorical("nu", [1.5, 2.5]) if kernel_type == "matern" else 1.5
        noise_floor = (1 - target_var_ratio) * 0.3
        noise_ceil  = (1 - target_var_ratio) * 1.5
        noise_level = trial.suggest_float("noise_level", noise_floor, noise_ceil, log=True)

        fold_mse        = []
        fold_ratio_gaps = []

        for train_idx, val_idx in folds:
            try:
                # Fresh scaler per fold - no leakage
                fold_scaler    = StandardScaler()
                y_train_scaled = fold_scaler.fit_transform(y[train_idx].reshape(-1, 1)).ravel()

                # signal_variance consistent with THIS fold's scaled space
                fold_signal_var = target_var_ratio * np.var(y_train_scaled, ddof=1)

                # Fresh kernel + GP per fold - no warm-starting across folds
                kernel = _build_kernel(kernel_type, length_scale, noise_level,
                                       fold_signal_var, nu, ls_min, ls_max)
                gp = GaussianProcessRegressor(
                    kernel=kernel, alpha=1e-6,
                    n_restarts_optimizer=0,   # Optuna drives search, not MLL
                    normalize_y=False,
                    random_state=42,
                )
                gp.fit(X[train_idx], y_train_scaled)

                # Validation MSE (prediction accuracy)
                preds_val = fold_scaler.inverse_transform(
                    gp.predict(X[val_idx]).reshape(-1, 1)
                ).ravel()
                fold_mse.append(np.mean((y[val_idx] - preds_val) ** 2))

                # Variance ratio on training fold - measures over-explanation
                # Training fold is correct here: we want the trend to obey the
                # budget on the data it was fitted to, not just unseen wells
                preds_train = fold_scaler.inverse_transform(
                    gp.predict(X[train_idx]).reshape(-1, 1)
                ).ravel()
                var_z_fold    = np.var(y[train_idx], ddof=1)
                var_pred_fold = np.var(preds_train, ddof=1)
                achieved_ratio = var_pred_fold / var_z_fold if var_z_fold > 1e-12 else 1.0
                fold_ratio_gaps.append(achieved_ratio - target_var_ratio)

            except Exception:
                return 1e6

        mean_mse       = float(np.mean(fold_mse))
        mean_ratio_gap = float(np.mean(fold_ratio_gaps))

        over_gap  = max(mean_ratio_gap, 0.0)   # positive when over-explaining
        under_gap = max(-mean_ratio_gap, 0.0)  # positive when under-explaining

        tolerance = 0.05

        # Only penalise over-explanation (GP absorbing too much residual
        # variance into the trend).  Under-explanation is left unpunished
        # so natural Cov(T,R) can emerge from scale overlap.
        if over_gap > tolerance:
            over_tolerance = 0.15
            if over_gap > over_tolerance:
                penalty = mean_mse * (0.5 + (over_gap / over_tolerance) ** 2)
            else:
                penalty = 0.0
        else:
            penalty = 0.0

        return mean_mse + penalty

    return objective




In [ ]:
def run_gp_for_dataset(samples, Z_samples, true_trend_values, grid_coords,
                        target_var_ratio, ny, nx,
                        n_trials=150, n_blocks_side=2, ls_max=900,
                        dataset_id=0, verbose=False, residual_range=200,
    ):
    """
    max_var_ratio : float, default 0.55
        Hard cap on the effective target variance ratio passed to the objective.
        Population-level ratios (e.g. 0.60) are at the upper edge of what is
        recommended for spatial trend models; capping at 0.55 gives the penalty
        more headroom and avoids passing an already-tight target to the optimizer.
    """
    X = samples.values if hasattr(samples, 'values') else np.array(samples)
    y = Z_samples.values if hasattr(Z_samples, 'values') else np.array(Z_samples)

    # Cap the target so the GP is not allowed to fill most of the variance budget
    rng = np.random.default_rng(seed=dataset_id)
    perturbation = rng.uniform(-0.05, 0.15)  # small random perturbation to avoid ties
    effective_target = np.clip(target_var_ratio + perturbation, 0.3, 0.83)

    final_scaler = StandardScaler()
    y_scaled = final_scaler.fit_transform(y.reshape(-1, 1)).ravel()

    # Var(y_scaled) â‰ˆ 1 by construction; signal_variance directly sets
    # the fraction of scaled variance attributed to trend
    signal_variance = effective_target  # since Var(y_scaled) â‰ˆ 1

    ls_min = residual_range * 0.5   # allow scale overlap with residuals

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name=f"gp_{dataset_id}",
    )
    study.optimize(
        make_constrained_objective(
            samples, Z_samples, effective_target,   # â† pass ratio, not pre-computed signal_variance
            n_blocks_side=n_blocks_side, ls_min=ls_min, ls_max=ls_max,
        ),
        n_trials=n_trials, show_progress_bar=True,
    )

    best = study.best_trial
    p = best.params
    nu = p.get("nu", 1.5)

    # Final refit: use CV-best params, NO additional MLL restarts
    # so the optimised hyperparameters are not silently overridden
    best_kernel = _build_kernel(
        p["kernel_type"], p["length_scale"], p["noise_level"],
        signal_variance, nu, ls_min, ls_max,
    )
    best_gp = GaussianProcessRegressor(
        kernel=best_kernel, alpha=1e-6,
        n_restarts_optimizer=0,   # honour CV result exactly
        normalize_y=False,
        random_state=42,
    )
    best_gp.fit(X, y_scaled)

    cov_pct, resid_pct, norm_rmse, point_trend, point_residual = compute_metrics(
        best_gp, samples, Z_samples, true_trend_values, scaler=final_scaler
    )

    actual_ratio = 1.0 - resid_pct
    ratio_gap = abs(actual_ratio - target_var_ratio)

    if verbose:
        print(f"  CV-MSE:              {best.value:.6f}")
        print(f"  Norm. RMSE:          {norm_rmse:.4f}")
        print(f"  Target Var(T)/Var(Z):{target_var_ratio:.0%}")
        print(f"  Actual Var(T)/Var(Z):{actual_ratio:.1%}")
        print(f"  Var(R)/Var(Z):       {resid_pct:.1%}")
        print(f"  2|Cov(T,R)|/Var(Z):  {cov_pct:.2%}")
        print(f"  Fitted kernel:       {best_gp.kernel_}")
        if ratio_gap > 0.15:
            print(f"  WARNING: ratio gap = {ratio_gap:.1%}")

    trend_grid = final_scaler.inverse_transform(
        best_gp.predict(grid_coords).reshape(-1, 1)
    ).ravel().reshape(ny, nx)

    return {
        "dataset_id": dataset_id,
        "cov_attribution_pct": cov_pct,
        "resid_pct": resid_pct,
        "normalized_rmse": norm_rmse,
        "point_trend": point_trend,
        "point_residual": point_residual,
        "model": best_gp,
        "trend_grid": trend_grid,
        "scaler": final_scaler,
        "effective_target": effective_target,
    }


### **Helper Function for Sequential Gaussian Simulation**

In [ ]:
from auto_variogram_tuner import *


def geostats_simulation(df, trend_grid, gs_trend_grid,
                         fallback_range=250.0, seed=2045,
                         n_realizations=25, nx=100, ny=100, dx=10.0,
                         verbose=False):
    """
    Run Sequential Gaussian Simulation (SGS) on residuals for two parallel tracks:

      No-GS   : simulates uncorrected residuals -> added back to original trend
      Post-GS : simulates GS-corrected residuals -> added back to corrected trend

    Parameters
    ----------
    df             : DataFrame with X, Y, Porosity, and residual columns.
                     If 'Wts_cell' is absent, declustering weights are computed
                     and attached to a working copy.
    trend_grid     : uncorrected trend grid, GSLIB convention (row 0 = y_max)
    gs_trend_grid  : GS-corrected trend grid, GSLIB convention
    fallback_range : range (m) for the fallback Spherical variogram
    seed           : base SGS seed; track B uses seed + 1
    n_realizations : number of SGS realizations per track
    nx, ny, dx     : simulation grid dimensions and cell size

    Returns
    -------
    dict    : keys {model_name}_hybrid_pre/post, _sgs_pre/post, _vdiag_pre/post
    df_copy : working copy of df with Wts_cell attached
    """


    col_pre  = "Estimated_Residual"
    col_post = "GS_Residual"

    # -- Declustering Weights (skip if already present) ----------------------
    df_copy = df.copy()
    if "Wts_cell" not in df_copy.columns:
        try:
            wts, _, _ = geostats.declus(df_copy, "X", "Y", "Porosity",
                                         iminmax=1, noff=10,
                                         ncell=100, cmin=1, cmax=5000)
        except IndexError:
            wts = np.ones(len(df_copy))
        df_copy["Wts_cell"] = wts
        df_copy[int(df_copy.columns.get_loc("Wts_cell"))] = wts

    # -- Automated Variogram Fitting -----------------------------------------
    vmodel_pre,  vdiag_pre  = auto_fit_variogram(df_copy, col_pre,  fallback_range=fallback_range)
    vmodel_post, vdiag_post = auto_fit_variogram(df_copy, col_post, fallback_range=fallback_range)

    # -- SGS Inner Function --------------------------------------------------
    def _sgsim(col, vmodel, seed_off):
        dat = df_copy[col]
        sig = dat.std()
        # Cap search radius to prevent memory blowup for long-range variograms
        MAX_RADIUS = 5 * dx * (20 ** 0.5)
        vm = dict(vmodel)
        if vm["hmaj1"] > MAX_RADIUS:
            vm["hmaj1"] = MAX_RADIUS
            vm["hmin1"] = min(vm["hmin1"], MAX_RADIUS)

        return geostats.sgsim(
            df_copy, "X", "Y", col,
            wcol=-1, scol=-1,
            tmin=-9999.9, tmax=9999.9,
            itrans=1, ismooth=0, dftrans=0, tcol=0, twtcol=0,
            zmin=dat.min() - 0.5 * sig,
            zmax=dat.max() + 0.5 * sig,
            ltail=2, ltpar=1.5,
            utail=2, utpar=1.5,
            nsim=n_realizations,
            nx=nx, xmn=dx / 2, xsiz=dx,
            ny=ny, ymn=dx / 2, ysiz=dx,
            seed=seed + seed_off,
            ndmin=0, ndmax=10, nodmax=10,
            mults=0, nmult=2, noct=-1,
            ktype=0,
            colocorr=0.0, sec_map=0,
            vario=vm,
        )

    sgs_pre  = _sgsim(col_pre,  vmodel_pre,  seed_off=0)
    sgs_post = _sgsim(col_post, vmodel_post, seed_off=1)

    # -- Reconstruct Porosity Realizations -----------------------------------
    hybrid_pre  = trend_grid[np.newaxis]    + sgs_pre
    hybrid_post = gs_trend_grid[np.newaxis] + sgs_post

    return {
        "hybrid_pre"  : hybrid_pre,
        "hybrid_post" : hybrid_post,
        "sgs_pre"     : sgs_pre,
        "sgs_post"    : sgs_post,
        "vdiag_pre"   : vdiag_pre,
        "vdiag_post"  : vdiag_post,
    }, df_copy

### **Full Pipeline For one Batch**

In [ ]:

# =============================================================================
# Master Pipeline: run_single_experiment(cfg)
# =============================================================================

def run_single_experiment(base_dir, n_sgs_realizations=50, n_gp_trials=100, use_wt_col=True,
                           nx=100, ny=100, dx=10.0, L=1000.0, dataset_name=None, verbose=False):
    """
    Run the full pipeline for one (trend_label, range_ratio, variance_ratio, fine_corr)
    configuration.

    Steps:
      1. Load Generated Data
      3. Fit GP trend model (spatial block CV, Optuna)
      4. Gram-Schmidt orthogonalization of residuals
      5-8. Variogram fitting, SGS (n_sgs_realizations per track), and hybrid
           construction via geostats_simulation()

    Returns generated grids and estimates.

    Convention note:
      Z_beta              -- standard convention (row 0 = y_min), as saved on disk.
      T_hat_grid, diff_kmap, sgs_*, hybrid_* -- GSLIB convention (row 0 = y_max),
        consistent with sgsim / kb2d outputs.
      When comparing hybrid grids to Z_beta (e.g. difference maps, RMSE), first
      call np.flipud() on the hybrid array to convert it to standard convention.
    """

    Z_beta = np.load(f'{base_dir}/{dataset_name}/Z_gaussian.npy')
    sampled_df = pd.read_csv(f'{base_dir}/{dataset_name}/sampled_wells.csv')

    with open(f"{base_dir}/{dataset_name}/metadata.json", "r") as f:
        meta_data = json.load(f)

    # -- Step 1: GP trend model -----------------------------------------------
    xs = np.linspace(dx / 2, L - dx / 2, nx)
    ys = np.linspace(dx / 2, L - dx / 2, ny)
    gx, gy = np.meshgrid(xs, ys)
    grid_coords = np.column_stack([gx.ravel(), gy.ravel()])

    offset_gpr_range = (meta_data['range_ratio'] * 1000.0) + 150

    dataset_id = meta_data['combo_index']
    gp_result = run_gp_for_dataset(
        samples=sampled_df[["X", "Y"]],
        Z_samples=sampled_df["Porosity"].values,
        true_trend_values=sampled_df["Trend_Por"].values,
        grid_coords=grid_coords,
        target_var_ratio=meta_data['actual_trend_to_total_var'],
        nx=nx, ny=ny,
        n_trials=n_gp_trials,
        n_blocks_side=2, ls_max=900,
        dataset_id=dataset_id, verbose=verbose, residual_range=offset_gpr_range,
    )

    sampled_df["Estimated_Trend"]    = gp_result["point_trend"]
    sampled_df["Estimated_Residual"] = gp_result["point_residual"]

    # -- Step 2: Gram-Schmidt orthogonalization --------------------------------
    _, point_trend_gs = gram_schmidt_orthogonalization(gp_result["point_residual"], gp_result["point_trend"])
    new_residuals = sampled_df['Porosity'].to_numpy() - point_trend_gs

    sampled_df['GS_Trend']       = point_trend_gs
    sampled_df['GS_Residual']    = new_residuals
    sampled_df['GS_adjustments'] = point_trend_gs - gp_result["point_trend"]

    # -- Step 2: Propagate GS trend adjustments to the full grid -------------
    diff_kmap     = smooth_krige_GS_adjustments(sampled_df, dx=dx, nx=nx, ny=ny)
    T_hat_grid    = np.flipud(gp_result["trend_grid"].reshape(ny, nx))
    T_hat_grid_gs = T_hat_grid + diff_kmap

    # -- Steps 4&5: Variogram fitting + SGS + hybrid construction ------------
    fallback_range = meta_data['range_ratio'] * 1000.0 / 2
    sgs_seed       = (dataset_id * 1000) + 45

    sim_result, sampled_df = geostats_simulation(
        sampled_df, T_hat_grid, T_hat_grid_gs,
        fallback_range=fallback_range, seed=sgs_seed,
        n_realizations=n_sgs_realizations,
        nx=nx, ny=ny, dx=dx,
    )

    sgs_A    = sim_result["sgs_pre"]
    sgs_B    = sim_result["sgs_post"]
    hybrid_A = sim_result["hybrid_pre"]
    hybrid_B = sim_result["hybrid_post"]
    vdiag_A  = sim_result["vdiag_pre"]
    vdiag_B  = sim_result["vdiag_post"]

    np.save(f"{base_dir}/{dataset_name}/T_hat_grid.npy",    T_hat_grid)
    np.save(f"{base_dir}/{dataset_name}/T_hat_grid_gs.npy", T_hat_grid_gs)
    np.save(f"{base_dir}/{dataset_name}/sgs_A.npy",         sgs_A)
    np.save(f"{base_dir}/{dataset_name}/sgs_B.npy",         sgs_B)
    np.save(f"{base_dir}/{dataset_name}/hybrid_A.npy",      hybrid_A)
    np.save(f"{base_dir}/{dataset_name}/hybrid_B.npy",      hybrid_B)
    np.save(f"{base_dir}/{dataset_name}/kriged_deltas.npy", diff_kmap)


    return {
        "dataset_id"      : int(dataset_id),
        "Z_beta"          : Z_beta,          # standard convention (row 0 = y_min)
        "hybrid_GS"       : hybrid_B,        # GSLIB convention   (row 0 = y_max)
        "hybrid"          : hybrid_A,        # GSLIB convention   (row 0 = y_max)
        "T_hat_grid"      : T_hat_grid,      # GSLIB convention   (row 0 = y_max)
        "T_hat_grid_gs"   : T_hat_grid_gs,
        "sgs_A"           : sgs_A,
        "sgs_B"           : sgs_B,
        "kriged_deltas"   : diff_kmap,
        "var_ratio"       : meta_data['actual_trend_to_total_var'],
        "vdiag_GS"        : vdiag_B,
        "vdiag_no_GS"     : vdiag_A,
    }, sampled_df

### **Diagnostic Visualization Helper Functions**

In [ ]:

def plot_reconstructed_fields(base_dir, results_, sampled_df, L, dataset_name, figsize=(20, 5)):
    Z_true      = results_["Z_beta"]
    kd          = np.flipud(results_["kriged_deltas"])
    trend_no_gs = np.flipud(results_["T_hat_grid"])
    trend_gs    = np.flipud(results_["T_hat_grid_gs"])

    norm_truth = Normalize(vmin=Z_true.min(),       vmax=Z_true.max())
    norm_trend = Normalize(vmin=min(trend_no_gs.min(), trend_gs.min()),
                           vmax=max(trend_no_gs.max(), trend_gs.max()))
    kd_lim  = max(abs(kd.min()), abs(kd.max()))
    norm_kd = TwoSlopeNorm(vmin=-kd_lim, vcenter=0.0, vmax=kd_lim)

    fig, axes = plt.subplots(1, 4, figsize=figsize)
    extent = [0, L, 0, L]

    def _imshow(ax, data, cmap, norm):
        return ax.imshow(data, origin="lower", extent=extent,
                         cmap=cmap, norm=norm, aspect="equal")

    def _scatter(ax, c, cmap, norm):
        ax.scatter(sampled_df["X"], sampled_df["Y"],
                   c=c, cmap=cmap, norm=norm,
                   s=15, edgecolors="k", linewidths=0.5, zorder=5)

    def _fmt(ax, title, ylabel=False):
        ax.set_title(title, fontsize=10, fontweight="bold", pad=4)
        ax.set_xlabel("Easting (m)", fontsize=9)
        if ylabel:
            ax.set_ylabel("Northing (m)", fontsize=9)
        ax.tick_params(labelsize=8)

    def _cbar(ax, im, label):
        # make_axes_locatable pins colorbar height to the actual axis height,
        # which is what fixes the oversized colorbar when aspect="equal" shrinks
        # the axes below the figure height.
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.06)
        cb = fig.colorbar(im, cax=cax)
        cb.set_label(label, fontsize=9)
        cb.ax.tick_params(labelsize=8)

    # Col 0: Ground Truth
    im0 = _imshow(axes[0], Z_true, "viridis", norm_truth)
    _scatter(axes[0], sampled_df.Porosity, "viridis", norm_truth)
    _fmt(axes[0], "Ground Truth", ylabel=True)
    _cbar(axes[0], im0, "Porosity")

    # Col 1: Predicted Trend (Without GS)
    im1 = _imshow(axes[1], trend_no_gs, "viridis", norm_trend)
    _scatter(axes[1], sampled_df.Estimated_Trend, "viridis", norm_trend)
    _fmt(axes[1], r"Predicted Trend (Without GS): $\hat{T}$")
    _cbar(axes[1], im1, "Porosity")

    # Col 2: Kriged GS Adjustments
    im2 = _imshow(axes[2], kd, "RdBu_r", norm_kd)
    _scatter(axes[2], sampled_df.GS_adjustments, "RdBu_r", norm_kd)
    _fmt(axes[2], "Kriged GS Adjustments")
    _cbar(axes[2], im2, "Adjustment")

    # Col 3: Predicted Trend (With GS)
    im3 = _imshow(axes[3], trend_gs, "viridis", norm_trend)
    _scatter(axes[3], sampled_df.GS_Trend, "viridis", norm_trend)
    _fmt(axes[3], r"Predicted Trend (With GS): $\hat{T}_{GS}$")
    _cbar(axes[3], im3, "Porosity")

    fig.suptitle("Reconstructed Fields: With vs Without GS Orthogonalization",
                 fontsize=12, fontweight="bold", y=1.01)

    plt.tight_layout()
    plt.savefig(f"{base_dir}/{dataset_name}/reconstructed_fields.png",
                dpi=900, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def plot_reproduction_check_hist_cdf(
    results_,
    sampled_df,
    nreal,
    *,
    data_col_no_gs="Estimated_Residual",
    data_col_gs="GS_Residual",
    weight_col="Wts_cell",
    sim_key_no_gs="sgs_A",
    sim_key_gs="sgs_B",
    bins=30,
    figsize=(16, 11),
    base_dir=None,
    dataset_name=None,
    save_path=None,
):
    """
    Histogram + CDF reproduction check: SGS residuals vs true residuals,
    for No GS and GS workflows side-by-side (rows).
    """

    def weighted_ecdf(x, w):
        x = np.asarray(x).ravel()
        w = np.asarray(w).ravel()
        mask = np.isfinite(x) & np.isfinite(w)
        x, w = x[mask], w[mask]
        idx = np.argsort(x)
        xs = x[idx]
        ws = w[idx].astype(float)
        ws /= ws.sum()
        return xs, np.cumsum(ws)

    def ecdf(x):
        x = np.asarray(x).ravel()
        x = x[np.isfinite(x)]
        xs = np.sort(x)
        return xs, np.arange(1, len(xs) + 1) / len(xs)

    #  Data prep 
    wts      = sampled_df[weight_col].to_numpy()
    wts_norm = wts / wts.sum()

    sim_no_gs  = np.asarray(results_[sim_key_no_gs])[:min(nreal, results_[sim_key_no_gs].shape[0])]
    sim_gs     = np.asarray(results_[sim_key_gs])[:min(nreal, results_[sim_key_gs].shape[0])]
    pool_no_gs = sim_no_gs.ravel()
    pool_gs    = sim_gs.ravel()

    data_no_gs = sampled_df[data_col_no_gs].to_numpy()
    data_gs    = sampled_df[data_col_gs].to_numpy()

    global_min = np.nanmin([data_no_gs.min(), data_gs.min(),
                            pool_no_gs.min(), pool_gs.min()])
    global_max = np.nanmax([data_no_gs.max(), data_gs.max(),
                            pool_no_gs.max(), pool_gs.max()])
    hist_bins  = np.linspace(global_min, global_max, bins) if np.isscalar(bins) else bins

    #  Typography constants 
    TITLE_FS    = 13
    LABEL_FS    = 12
    TICK_FS     = 11.5
    LEGEND_FS   = 11.5
    SUPTITLE_FS = 14

    #  Figure with explicit GridSpec spacing 
    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(
        2, 2,
        figure=fig,
        hspace=0.38,   # vertical gap between rows
        wspace=0.28,   # horizontal gap between histogram and CDF columns
        top=0.91, bottom=0.08,
        left=0.07, right=0.97,
    )

    axes = np.empty((2, 2), dtype=object)
    for r in range(2):
        for c in range(2):
            axes[r, c] = fig.add_subplot(gs[r, c])

    #  Plot loop 
    workflows = [
        ("Without GS", data_no_gs, sim_no_gs, pool_no_gs, axes[0, 0], axes[0, 1]),
        ("With GS",    data_gs,    sim_gs,    pool_gs,    axes[1, 0], axes[1, 1]),
    ]

    # Colour palette  distinct, print-safe
    CLR_DATA   = "#2166AC"   # blue   declustered data
    CLR_POOL   = "#D6604D"   # red    pooled realizations
    CLR_REAL   = "#AAAAAA"   # grey   individual realizations
    CLR_DATA_H = "#4393C3"   # lighter blue for histogram fill
    CLR_POOL_H = "#F4A582"   # lighter red  for histogram fill

    for label, data, sims, sim_pool, ax_h, ax_c in workflows:

        #  Histogram 
        ax_h.hist(data, bins=hist_bins, density=True, weights=wts_norm,
                  color=CLR_DATA_H, alpha=0.75, edgecolor="white",
                  linewidth=0.6, label="Declustered Data", zorder=3)
        ax_h.hist(sim_pool, bins=hist_bins, density=True,
                  color=CLR_POOL_H, alpha=0.65, edgecolor="white",
                  linewidth=0.6, label="All Realizations", zorder=2)

        ax_h.set_title(f"Histogram  {label}", fontweight="bold",
                       fontsize=TITLE_FS, pad=6)
        ax_h.set_xlabel("Residual", fontsize=LABEL_FS, labelpad=4)
        ax_h.set_ylabel("Density",  fontsize=LABEL_FS, labelpad=4)
        ax_h.tick_params(labelsize=TICK_FS)
        ax_h.legend(fontsize=LEGEND_FS, framealpha=0.9,
                    edgecolor="grey", loc="upper right")
        ax_h.grid(True, linestyle=":", alpha=0.4, linewidth=0.7)
        ax_h.set_xlim(global_min, global_max)

        #  CDF 
        for i, real in enumerate(sims):
            sx, sy = ecdf(real.ravel())
            ax_c.plot(sx, sy, color=CLR_REAL, alpha=0.9, lw=1.8, zorder=2,
                      label="Realization" if i == 0 else None)

        sx_pool, sy_pool = ecdf(sim_pool)
        ax_c.plot(sx_pool, sy_pool, color=CLR_POOL, lw=2.2, alpha=0.9,
                  label="Pooled Realizations", zorder=3)

        x_data, y_data = weighted_ecdf(data, wts)
        ax_c.plot(x_data, y_data, color=CLR_DATA, lw=2.5,
                  label="Declustered Data", zorder=4)

        ax_c.set_title(f"CDF  {label}", fontweight="bold",
                       fontsize=TITLE_FS, pad=6)
        ax_c.set_xlabel("Residual",               fontsize=LABEL_FS, labelpad=4)
        ax_c.set_ylabel("Cumulative Probability", fontsize=LABEL_FS, labelpad=4)
        ax_c.set_xlim(global_min, global_max)
        ax_c.set_ylim(0, 1)
        ax_c.tick_params(labelsize=TICK_FS)
        ax_c.grid(True, linestyle=":", alpha=0.4, linewidth=0.7)

        # Legend: place outside to the right so it never clips the curves
        ax_c.legend(
            fontsize=LEGEND_FS, framealpha=0.9, edgecolor="grey",
            loc="upper left", bbox_to_anchor=(1.02, 1.0),
            borderpad=0.6, handlelength=1.6,
        )

    #  Suptitle 
    fig.suptitle(
        "Histogram & CDF Reproduction Check  SGS Residuals (Without GS vs With GS)",
        fontsize=SUPTITLE_FS, fontweight="bold", y=0.97,
    )

    #  Save 
    out_path = save_path
    if out_path is None and base_dir is not None and dataset_name is not None:
        out_path = f"{base_dir}/{dataset_name}/repro_hist_cdf.png"
    if out_path is not None:
        fig.savefig(out_path, dpi=1200, bbox_inches="tight", facecolor="white")


In [ ]:
def plot_residual_variograms(sampled_df, dataset_name,base_dir,
                              xlag=25, xltol=22.5, nlag=40,
                              pair_threshold=100, figsize=(12, 5)):
    """
    Compute and plot experimental variograms for:
      - Ground Truth Residuals
      - GPR Residuals (No GS)
      - GPR + GS Residuals

    Parameters
    ----------
    sampled_df      : DataFrame with columns X, Y, Res_Por, Estimated_Residual, GS_Residual
    dataset_name    : str, used for save path
    geostats        : geostats module
    xlag            : float, lag spacing in meters
    xltol           : float, lag tolerance
    nlag            : int, number of lags
    pair_threshold  : int, minimum pairs per lag bin to include
    figsize         : tuple
    """
    gamv_kwargs = dict(
        tmin=-999, tmax=999,
        xlag=xlag, xltol=xltol, nlag=nlag,
        azm=0, atol=90, bandwh=9999.9, isill=0,
    )

    #  Compute variograms 
    res_lags,      res_var,      res_pairs      = geostats.gamv(sampled_df, "X", "Y", "Estimated_Residual", **gamv_kwargs)
    true_res_lags, true_res_var, true_res_pairs = geostats.gamv(sampled_df, "X", "Y", "Res_Por",            **gamv_kwargs)
    gs_res_lags,   gs_res_var,   gs_res_pairs   = geostats.gamv(sampled_df, "X", "Y", "GS_Residual",        **gamv_kwargs)

    #  Masks 
    mask      = (res_pairs      > pair_threshold) & np.isfinite(res_lags)      & np.isfinite(res_var)
    mask_true = (true_res_pairs > pair_threshold) & np.isfinite(true_res_lags) & np.isfinite(true_res_var)
    mask_gs   = (gs_res_pairs   > pair_threshold) & np.isfinite(gs_res_lags)   & np.isfinite(gs_res_var)

    #  Plot 
    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(true_res_lags[mask_true], true_res_var[mask_true],
               color="green", edgecolor="black", s=50, zorder=8,
               label="Ground Truth Residuals")

    ax.scatter(res_lags[mask], res_var[mask],
               color="orange", edgecolor="black", s=50, zorder=8,
               label="GPR Residuals (Without GS)")

    ax.scatter(gs_res_lags[mask_gs], gs_res_var[mask_gs],
               color="blue", edgecolor="black", s=50, zorder=8,
               label="GPR Residuals (With GS)")

    ax.set_xlabel(r'Lag Distance $\bf(h)$, (m)', fontsize=11)
    ax.set_ylabel(r'$\gamma \bf(h)$',            fontsize=11)
    ax.set_title("Residual Variogram Comparison",fontsize=12, fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout(rect=[0, 0, 0.82, 1])  # leave room for legend on right
    plt.savefig(f"{base_dir}/{dataset_name}/residual_variograms.png",
                dpi=1200, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def plot_variance_decomposition(point_trend_A, point_residual_A,
                                 point_trend_B, point_residual_B,
                                 Z_samples, colors=None,
                                 titles=None, dataset_name=None,base_dir=None):
    """
    Side-by-side variance decomposition: Before GS (left) vs After GS (right).
    Uses pie chart if Cov >= 0, signed bar chart if Cov < 0.
    Suppresses covariance slice in pie chart if abs(cov) < 0.05%.

    Parameters
    ----------
    point_trend_A    : array-like, trend values before GS at sample locations
    point_residual_A : array-like, residual values before GS at sample locations
    point_trend_B    : array-like, trend values after GS at sample locations
    point_residual_B : array-like, residual values after GS at sample locations
    Z_samples        : array-like, observed Z values at sample locations
    colors           : list of 3 colors [Trend, Residual, Covariance]
    fig, axes        : optional existing figure and 1x2 axes array
    titles           : list of 2 strings, default ['Before GS', 'After GS']
    """
    if colors is None:
        colors = ['#CC5801', '#66b3ff', '#FFFF66']
    if titles is None:
        titles = ['Before GS Orthogonalization', 'After GS Orthogonalization']
    fig, axes = plt.subplots(1, 2, figsize=(8, 5))

    Z_samples = np.asarray(Z_samples)
    var_z     = float(np.var(Z_samples, ddof=1))

    datasets = [
        (point_trend_A, point_residual_A),
        (point_trend_B, point_residual_B),
    ]

    for ax, (trend, residual), title in zip(axes, datasets, titles):
        trend    = np.asarray(trend)
        residual = np.asarray(residual)

        var_t   = float(np.var(trend,    ddof=1))
        var_r   = float(np.var(residual, ddof=1))
        cov_tr  = var_z - var_t - var_r       # = 2*Cov(T,R)

        pct_t   = var_t  / var_z * 100
        pct_r   = var_r  / var_z * 100
        pct_cov = cov_tr / var_z * 100        # signed

        ax.set_title(f"{title}\nCov(T,R) = {pct_cov:.1f}%",
                     fontsize=10, fontweight='bold')

        if pct_cov >= 0:
            #  Pie chart 
            sizes      = [pct_t, pct_r]
            labels     = ['Trend', 'Residual']
            pie_colors = [colors[0], colors[1]]

            # only add covariance slice if meaningful
            if abs(pct_cov) >= 0.05:
                sizes.append(pct_cov)
                labels.append('Covariance')
                pie_colors.append(colors[2])

            wedges, texts, autotexts = ax.pie(
                sizes,
                labels=labels,
                colors=pie_colors,
                autopct='%1.1f%%',
                startangle=90,
                wedgeprops=dict(edgecolor='white', linewidth=1.2)
            )
            for at in autotexts:
                at.set_fontsize(9)
            for t in texts:
                t.set_fontsize(9)

        else:
            #  Signed stacked bar chart 
            bar_labels  = ['Trend', 'Residual', 'Covariance', 'Total']
            values      = [pct_t, pct_r, pct_cov, 100.0]
            bar_colors  = colors + ['#888888']
            bar_heights = [pct_t, pct_r, abs(pct_cov), 100.0]
            bar_bottoms = [0,     0,     min(0, pct_cov), 0]

            bars = ax.bar(
                bar_labels,
                bar_heights,
                bottom=bar_bottoms,
                color=bar_colors,
                edgecolor='white',
                linewidth=1.2,
                width=0.5
            )

            for bar, val, bottom in zip(bars, values, bar_bottoms):
                y_pos = bottom + bar.get_height() / 2
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f'{val:.1f}%',
                    ha='center', va='center',
                    fontsize=9, fontweight='bold', color='black'
                )

            ax.axhline(100, color='black', linewidth=1.0,
                       linestyle='--', alpha=0.5)
            ax.axhline(0,   color='black', linewidth=0.8,
                       linestyle='-',  alpha=0.3)
            ax.set_ylabel('% of Total Variance', fontsize=9)
            ax.set_ylim(min(0, pct_cov) - 10, 115)
            ax.tick_params(axis='both', labelsize=9)
            ax.spines[['top', 'right']].set_visible(False)
            ax.text(
                0.98, 0.97,
                'Negative Cov(T,R):\nVar(T)+Var(R) > Var(Z)',
                transform=ax.transAxes,
                ha='right', va='top',
                fontsize=7.5, color='gray', style='italic'
            )

    fig.suptitle('Variance Decomposition: Before vs After GS Orthogonalization',
                 fontsize=12, fontweight='bold', y=1.02)
    fig.tight_layout()
    plt.savefig(f"{base_dir}/{dataset_name}/variance_decomposition.png", dpi=1200, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def plot_reproduction_check_scatter(results_, sampled_df, base_dir, dataset_name, n_real, nx=100, ny=100, dx=10.0):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Left: No GS ---
    plt.sca(axes[0])

    df_vstack_A = GSLIB.conditioning_check(
        model=results_['hybrid'],
        nx=nx, xmn=dx/2, xsiz=dx,
        ny=ny, ymn=dx/2, ysiz=dx,
        nreal=n_real,
        df=sampled_df,
        xcol='X', ycol='Y',
        vcol='Porosity', vname='Porosity',
        vmin=results_['hybrid'].min(),
        vmax=results_['hybrid'].max()
    )

    sns.kdeplot(
        data=df_vstack_A,
        x="Data", y="Real",
        color="black", alpha=0.4
    )

    axes[0].set_title("Conditioning Check - Hybrid (Without GS)")
    axes[0].set_xlabel("Data")
    axes[0].set_ylabel("Realization")


    # --- Right: With GS ---
    plt.sca(axes[1])

    df_vstack_B = GSLIB.conditioning_check(
        model=results_['hybrid_GS'],
        nx=nx, xmn=dx/2, xsiz=dx,
        ny=ny, ymn=dx/2, ysiz=dx,
        nreal=n_real,
        df=sampled_df,
        xcol='X', ycol='Y',
        vcol='Porosity', vname='Porosity',
        vmin=results_['hybrid_GS'].min(),
        vmax=results_['hybrid_GS'].max()
    )

    sns.kdeplot(
        data=df_vstack_B,
        x="Data", y="Real",
        color="black", alpha=0.4
    )

    axes[1].set_title("Conditioning Check - Hybrid (With GS)")
    axes[1].set_xlabel("Data")
    axes[1].set_ylabel("Realization")

    plt.tight_layout()
    plt.savefig(f"{base_dir}/{dataset_name}/reproduction_check_scatter.png", dpi=1200, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def compute_coverage_curve(Z_true, hybrid, nominal_levels=None):
    """
    Compute actual coverage at each nominal probability level.

    Parameters
    ----------
    Z_true : np.ndarray, shape (ny, nx)
        Ground truth grid.
    hybrid : np.ndarray, shape (n_real, ny, nx)
        Ensemble of realizations.
    nominal_levels : array-like, optional
        Nominal coverage levels between 0 and 1 (e.g. [0.1, 0.2, ..., 0.9]).
        Default: np.arange(0.05, 1.0, 0.05) - 5% to 95% in steps of 5%.

    Returns
    -------
    nominal_levels : np.ndarray
        The nominal coverage levels used.
    actual_coverage : np.ndarray
        Fraction of grid nodes where Z_true falls within the symmetric interval.
    """
    if nominal_levels is None:
        nominal_levels = np.arange(0.02, 1.0, 0.02)
    nominal_levels = np.asarray(nominal_levels)

    n_nodes = Z_true.size
    z_flat = Z_true.ravel()                         # (n_nodes,)
    h_flat = hybrid.reshape(hybrid.shape[0], -1)    # (n_real, n_nodes)

    actual_coverage = np.zeros(len(nominal_levels))

    for i, p in enumerate(nominal_levels):
        lower_q = (1.0 - p) / 2.0 * 100.0   # e.g. p=0.80  lower=10th percentile
        upper_q = (1.0 + p) / 2.0 * 100.0   # e.g. p=0.80  upper=90th percentile

        lower_bound = np.percentile(h_flat, lower_q, axis=0)  # (n_nodes,)
        upper_bound = np.percentile(h_flat, upper_q, axis=0)  # (n_nodes,)

        within = (z_flat >= lower_bound) & (z_flat <= upper_bound)
        actual_coverage[i] = np.sum(within) / n_nodes

    return nominal_levels, actual_coverage


def deutsch_goodness(nominal_levels, actual_coverage):
    """
    Compute the Deutsch (1997) Goodness statistic G using the asymmetric
    penalty formulation from "Direct Assessment of Local Accuracy and Precision".

    G = 1 - integral_0^1 w(p) * |xi(p) - p| dp  /  integral_0^1 w(p) * max_dev(p) dp

    where:
        xi(p) = actual coverage at nominal level p
        a(p)  = 1 if xi(p) >= p  (overcoverage / conservative)
        a(p)  = 0 if xi(p) <  p  (undercoverage / overconfident)

    Penalty weights:
        - Overcoverage (above 45Â° line): w(p) = 1
        - Undercoverage (below 45Â° line): w(p) = -2

    The denominator normalizes by the worst-case penalty so G stays in [0, 1].

    Undercoverage is penalized twice as heavily because overconfident
    predictions (intervals too narrow) are worse than conservative ones.

    Parameters
    ----------
    nominal_levels : array-like
        Nominal coverage levels (between 0 and 1).
    actual_coverage : array-like
        Corresponding actual coverage fractions.

    Returns
    -------
    G : float
        Deutsch goodness statistic.
    """
    nominal_levels = np.asarray(nominal_levels, dtype=float)
    actual_coverage = np.asarray(actual_coverage, dtype=float)

    overcoverage = actual_coverage >= nominal_levels  # I_0(p)

    # [3*I_0(p) - 2] * [kappa_bar(p) - p]
    weight = np.where(overcoverage, 1.0, -2.0)          # 3*I_0 - 2
    deviation = actual_coverage - nominal_levels         # kappa_bar - p (signed)
    integrand = weight * deviation                       # always >= 0

    penalty_integral = np.trapz(integrand, nominal_levels)
    G = 1.0 - penalty_integral

    return G


def goodness_plot(Z_true, hybrid_A, hybrid_B, base_dir, dataset_name,
                  label_A="Without GS", label_B="With GS",
                  nominal_levels=None, figsize=(8, 7),
                  title="Uncertainty Goodness Plot: Without GS vs With GS",
                  save_path=None):
    """
    Produce the accuracy (goodness) plot comparing two workstreams
    and report the Deutsch Goodness statistic for each.

    Parameters
    ----------
    Z_true : np.ndarray, shape (ny, nx)
        Ground truth grid.
    hybrid_A : np.ndarray, shape (n_real, ny, nx)
        Realizations from Workstream A (e.g. No GS).
    hybrid_B : np.ndarray, shape (n_real, ny, nx)
        Realizations from Workstream B (e.g. GS).
    label_A, label_B : str
        Legend labels for each workstream.
    nominal_levels : array-like, optional
        Nominal coverage levels. Default: 5% to 95% in steps of 5%.
    figsize : tuple
        Figure size.
    title : str
        Plot title.
    save_path : str or None
        If provided, save the figure to this path.

    Returns
    -------
    results : dict with keys:
        'nominal'    : nominal coverage levels
        'actual_A'   : actual coverage for Workstream A
        'actual_B'   : actual coverage for Workstream B
        'G_A'        : Deutsch Goodness for Workstream A
        'G_B'        : Deutsch Goodness for Workstream B
    """
    nom_A, act_A = compute_coverage_curve(Z_true, hybrid_A, nominal_levels)
    nom_B, act_B = compute_coverage_curve(Z_true, hybrid_B, nominal_levels)

    G_A = deutsch_goodness(nom_A, act_A)
    G_B = deutsch_goodness(nom_B, act_B)

    #  Plot 
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # 45-degree reference
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.0, label='Perfect Calibration')

    # Workstream curves
    ax.plot(nom_A, act_A, 'o-', color='#E74C3C', linewidth=2, markersize=5,
            label=f'{label_A}  (G = {G_A:.3f})')
    ax.plot(nom_B, act_B, 's-', color='#2E86C1', linewidth=2, markersize=5,
            label=f'{label_B}  (G = {G_B:.3f})')

    # Shade deviation from ideal for each workstream
    ax.fill_between(nom_A, nom_A, act_A, color='#E74C3C', alpha=0.08)
    ax.fill_between(nom_B, nom_B, act_B, color='#2E86C1', alpha=0.08)

    ax.set_xlabel('Nominal Coverage', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual Coverage', fontsize=13, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='upper left')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # Annotation
    ax.text(0.95, 0.05,
            f'Goodness score:\n{label_A}: G = {G_A:.3f}\n{label_B}: G = {G_B:.3f}',
            transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat', alpha=0.5))

    plt.tight_layout()

    save_path = f"{base_dir}/{dataset_name}/goodness_plot.png"
    fig.savefig(save_path, dpi=1200, bbox_inches='tight')
    print(f"Figure saved to {save_path}")

    return {
        'nominal': nom_A,
        'actual_A': act_A,
        'actual_B': act_B,
        'G_A': G_A,
        'G_B': G_B,
    }


In [ ]:
def prepare_summary_maps(results_, L):
    """
    Prepare summary maps for Without GS and With GS workflows from a results_ dict.

    Convention
    ----------
    Z_beta is in standard convention (row 0 = y_min).
    sgs_A/B and T_hat_grid/T_hat_grid_gs are in GSLIB convention (row 0 = y_max).
    The hybrid stacks are flipped to standard convention so all arrays
    are consistent with Z_beta and display correctly with pcolormesh
    using y = linspace(0, L, ny).

    Parameters
    ----------
    results_ : dict
        Expected keys:
            sgs_A         (nreal, ny, nx)  -- Without GS residual realizations (GSLIB)
            sgs_B         (nreal, ny, nx)  -- With GS residual realizations (GSLIB)
            T_hat_grid    (ny, nx)         -- Without GS trend grid (GSLIB)
            T_hat_grid_gs (ny, nx)         -- With GS trend grid (GSLIB)
            Z_beta        (ny, nx)         -- truth, standard convention
    L : float
        Domain extent.

    Returns
    -------
    prepared : dict
        Keys "Without GS" and "With GS", each with summary maps, plus "_meta".
        All 2-D arrays are in standard convention (row 0 = y_min).
    """
    required = ["sgs_A", "sgs_B", "T_hat_grid", "T_hat_grid_gs", "Z_beta"]
    missing = [k for k in required if k not in results_]
    if missing:
        raise KeyError(f"Missing required keys in results_: {missing}")

    # Z_beta is already standard convention - no flip needed
    z_truth = np.asarray(results_["Z_beta"])

    ny, nx = z_truth.shape
    x = np.linspace(0, L, nx)
    y = np.linspace(0, L, ny)

    tracks = {
        "Without GS": (
            np.asarray(results_["sgs_A"]),
            np.asarray(results_["T_hat_grid"]),
        ),
        "With GS": (
            np.asarray(results_["sgs_B"]),
            np.asarray(results_["T_hat_grid_gs"]),
        ),
    }

    prepared = {}
    for label, (sgs, trend_grid) in tracks.items():
        if sgs.ndim != 3:
            raise ValueError(
                f"{label}: sgs must have shape (nreal, ny, nx), got {sgs.shape}"
            )

        # Build hybrid (GSLIB convention) then flip to standard convention
        stack = sgs + trend_grid[np.newaxis, :, :]   # (nreal, ny, nx), GSLIB
        stack = stack[:, ::-1, :]                     # â†’ standard convention (row 0 = y_min)

        if stack.shape[1:] != z_truth.shape:
            raise ValueError(
                f"{label}: hybrid shape {stack.shape[1:]} does not match z_truth {z_truth.shape}"
            )

        e_type = np.mean(stack, axis=0)
        std_map = np.std(stack, axis=0)
        p10 = np.percentile(stack, 10, axis=0)
        p50 = np.percentile(stack, 50, axis=0)
        p90 = np.percentile(stack, 90, axis=0)

        within_p80 = ((z_truth >= p10) & (z_truth <= p90)).astype(float)

        error = e_type - z_truth
        abs_error = np.abs(error)

        prepared[label] = {
            "stack": stack,
            "e_type": e_type,
            "std_map": std_map,
            "p10": p10,
            "p50": p50,
            "p90": p90,
            "within_p80": within_p80,
            "coverage": float(within_p80.mean()),
            "error": error,
            "abs_error": abs_error,
        }

    prepared["_meta"] = {"x": x, "y": y, "z_truth": z_truth, "L": L}
    return prepared


def _make_mesh(prepared):
    x = prepared["_meta"]["x"]
    y = prepared["_meta"]["y"]
    return np.meshgrid(x, y)


def _ensure_dir(save_path):
    if save_path is not None:
        directory = os.path.dirname(save_path)
        if directory:
            os.makedirs(directory, exist_ok=True)

In [ ]:
def plot_gs_comparison(prepared, sampled_df=None,
                       figsize=(20, 13), save_path=None):
    """
    Publication-quality 2x 3 comparison figure.

    Layout
    ------
    Row 0:  Ground Truth  |  E-type No GS  |  E-type GS     â† shared colorbar on right
    Row 1:  Coverage No GS | Coverage GS   | Goodness Plot

    Design choices
    --------------
    * Single shared colorbar placed outside the map columns (right side, row 0).
    * Coverage legend placed BELOW each coverage map (outside the ).
    * Goodness-plot legend placed outside the  (upper-left, bbox_to_anchor).
    * Font sizes chosen for a two-column journal figure (scale with figsize).
    * 600 dpi output, tight bbox.
    """

    # Data Unpacking
    X, Y   = _make_mesh(prepared)
    L      = prepared["_meta"]["L"]
    z_truth = prepared["_meta"]["z_truth"]
    no_gs  = prepared["Without GS"]
    gs     = prepared["With GS"]

    # Shared porosity colour scale
    all_vals = np.concatenate([z_truth.ravel(),
                               no_gs["e_type"].ravel(),
                               gs["e_type"].ravel()])
    vmin_p, vmax_p = np.nanpercentile(all_vals, 1), np.nanpercentile(all_vals, 99)

    #  Figure & GridSpec 
    # Use constrained_layout=False so we drive every spacing manually.
    fig = plt.figure(figsize=figsize, constrained_layout=False)

    # Two rows; give row 1 (coverage + goodness) slightly more height
    outer = gridspec.GridSpec(
        2, 1, figure=fig,
        hspace=0.38,          # vertical gap between rows
        top=0.91, bottom=0.07,
        left=0.06, right=0.88,
    )

    # Row 0: three map panels + colorbar column
    gs_top = gridspec.GridSpecFromSubplotSpec(
        1, 3, subplot_spec=outer[0],
        wspace=0.08,
    )

    # Row 1: two coverage panels + goodness panel
    gs_bot = gridspec.GridSpecFromSubplotSpec(
        1, 3, subplot_spec=outer[1],
        wspace=0.32,          # more space so goodness plot breathes
    )

    ax_truth  = fig.add_subplot(gs_top[0])
    ax_nogs_e = fig.add_subplot(gs_top[1])
    ax_gs_e   = fig.add_subplot(gs_top[2])

    ax_nogs_c = fig.add_subplot(gs_bot[0])
    ax_gs_c   = fig.add_subplot(gs_bot[1])
    ax_good   = fig.add_subplot(gs_bot[2])

    TITLE_FS  = 12.5
    LABEL_FS  = 11.5
    TICK_FS   = 11.5
    LEGEND_FS = 11.5
    SUPTITLE_FS = 15

    def _fmt_map(ax, title):
        ax.set_title(title, fontweight="bold", fontsize=TITLE_FS, pad=6)
        ax.set_xlabel("Easting (m)",  fontsize=LABEL_FS, labelpad=4,)
        ax.set_ylabel("Northing (m)", fontsize=LABEL_FS, labelpad=4)
        ax.tick_params(labelsize=TICK_FS)
        ax.set_aspect("equal")
        ax.set_xlim(0, L)
        ax.set_ylim(0, L)
        ax.xaxis.set_major_locator(MultipleLocator(200))
        ax.yaxis.set_major_locator(MultipleLocator(200))

    def _well_scatter(ax, c=None, vmin=None, vmax=None):
        if sampled_df is None:
            return
        kw = dict(s=18, edgecolors="k", linewidths=0.4, zorder=5)
        if c is not None:
            ax.scatter(sampled_df["X"], sampled_df["Y"],
                       c=c, cmap="inferno",
                       vmin=vmin, vmax=vmax, **kw)
        else:
            ax.scatter(sampled_df["X"], sampled_df["Y"],
                       color="white", **kw)

    #  Row 0: Porosity maps 
    pcolor_kw = dict(shading="auto", cmap="inferno",
                     vmin=vmin_p, vmax=vmax_p, rasterized=True)

    im = ax_truth.pcolormesh(X, Y, z_truth, **pcolor_kw)
    _well_scatter(ax_truth,
                  c=sampled_df["Porosity"] if sampled_df is not None else None,
                  vmin=vmin_p, vmax=vmax_p)
    _fmt_map(ax_truth, "Ground Truth")

    ax_nogs_e.pcolormesh(X, Y, no_gs["e_type"], **pcolor_kw)
    _well_scatter(ax_nogs_e,
                  c=sampled_df["Porosity"] if sampled_df is not None else None,
                  vmin=vmin_p, vmax=vmax_p)
    _fmt_map(ax_nogs_e, "E-type  Without GS")
    ax_nogs_e.set_ylabel("")          # suppress redundant y-label on middle panel

    ax_gs_e.pcolormesh(X, Y, gs["e_type"], **pcolor_kw)
    _well_scatter(ax_gs_e,
                  c=sampled_df["Porosity"] if sampled_df is not None else None,
                  vmin=vmin_p, vmax=vmax_p)
    _fmt_map(ax_gs_e, "E-type  GS")
    ax_gs_e.set_ylabel("")

    # Shared colorbar  placed in the right margin, aligned with row 0
    cbar_ax = fig.add_axes([0.895, outer[0].get_position(fig).y0,
                             0.018,  outer[0].get_position(fig).height])
    cb = fig.colorbar(im, cax=cbar_ax)
    cb.set_label("Value", fontsize=LABEL_FS, labelpad=8)
    cb.ax.tick_params(labelsize=TICK_FS)

    #  Row 1 left/centre: Coverage maps 
    cov_kw = dict(shading="auto", cmap="RdYlGn",
                  vmin=0, vmax=1, rasterized=True)

    ax_nogs_c.pcolormesh(X, Y, no_gs["within_p80"], **cov_kw)
    _well_scatter(ax_nogs_c)
    _fmt_map(ax_nogs_c,
             f"P10-P90 Coverage: {no_gs['coverage']:.1%}  Without GS")

    ax_gs_c.pcolormesh(X, Y, gs["within_p80"], **cov_kw)
    _well_scatter(ax_gs_c)
    _fmt_map(ax_gs_c,
             f"P10-P90 Coverage: {gs['coverage']:.1%}  With GS")
    ax_gs_c.set_ylabel("")

    # Coverage legend  placed BELOW the two coverage panels, centred
    cov_handles = [
        Patch(facecolor="#1a9641", edgecolor="k", linewidth=0.5,
              label="Within P10-P90"),
        Patch(facecolor="#d7191c", edgecolor="k", linewidth=0.5,
              label="Outside P10-P90"),
    ]
    # Attach to the left coverage panel; place below it via bbox_to_anchor
    ax_nogs_c.legend(
        handles=cov_handles,
        loc="upper center",
        bbox_to_anchor=(1.05, -0.18),   # centred between the two coverage 
        ncol=2,
        fontsize=LEGEND_FS,
        framealpha=0.9,
        edgecolor="grey",
        handlelength=1.4,
    )

    #  Row 1 right: Goodness plot 
    nom_A, act_A = compute_coverage_curve(z_truth, no_gs["stack"])
    nom_B, act_B = compute_coverage_curve(z_truth, gs["stack"])
    G_A = deutsch_goodness(nom_A, act_A)
    G_B = deutsch_goodness(nom_B, act_B)
    # Anchor at (0, 0): coverage at p=0 is analytically 0
    nom_A, act_A = np.r_[0, nom_A], np.r_[0, act_A]
    nom_B, act_B = np.r_[0, nom_B], np.r_[0, act_B]

    ax_good.plot([0, 1], [0, 1], "k--", linewidth=1.2,
                 label="Perfect calibration", zorder=2)
    ax_good.plot(nom_A, act_A, "o-", color="#E74C3C",
                 linewidth=2, markersize=5,
                 label=f"Without GS  (G = {G_A:.3f})", zorder=3)
    ax_good.plot(nom_B, act_B, "s-", color="#2E86C1",
                 linewidth=2, markersize=5,
                 label=f"With GS  (G = {G_B:.3f})", zorder=3)

    ax_good.set_xlabel("Nominal Coverage (p)", fontsize=LABEL_FS, labelpad=4)
    ax_good.set_ylabel("Actual Coverage  \u03ba\u0305(p)", fontsize=LABEL_FS, labelpad=4)
    ax_good.set_title("Uncertainty Goodness Plot", fontweight="bold", fontsize=TITLE_FS, pad=6)
    ax_good.set_xlim(0, 1)
    ax_good.set_ylim(0, 1)
    ax_good.set_aspect("equal")
    ax_good.tick_params(labelsize=TICK_FS)
    ax_good.xaxis.set_major_locator(MultipleLocator(0.2))
    ax_good.yaxis.set_major_locator(MultipleLocator(0.2))
    ax_good.grid(True, alpha=0.3, linewidth=0.6)

    # Legend outside the goodness   to the right
    ax_good.legend(
        loc="upper left",
        bbox_to_anchor=(1.04, 1.0),
        fontsize=LEGEND_FS,
        framealpha=0.9,
        edgecolor="grey",
        handlelength=1.8,
        borderpad=0.7,
    )

    #  Suptitle 
    fig.suptitle("With GS vs Without GS Workflow Comparison",
                 fontweight="bold", fontsize=SUPTITLE_FS, y=0.97)

    #  Save / show 
    if save_path:
        _ensure_dir(save_path)
        fig.savefig(save_path, dpi=1200, bbox_inches="tight", facecolor="white")
        print(f"Figure saved â†’ {save_path}")

    plt.show()

### **Metrics**

In [ ]:
def pct(arr, q): return np.percentile(arr, q)

def get_metrics(results_, sampled_df, L, full_at_wells, full_at_wells_gs):
    """
    Compute grid-level and sample-level metrics for No GS and GS hybrid models.

    Parameters
    ----------
    results_         : dict with keys: Z_beta, hybrid, hybrid_GS,
                       T_hat_grid, T_hat_grid_gs, vdiag_no_GS, vdiag_GS
    sampled_df       : DataFrame with columns: X, Y, Porosity,
                       Estimated_Trend, Estimated_Residual, GS_Trend, GS_Residual
    L                : float, domain length in meters
    full_at_wells    : np.ndarray (n_real, n_wells)  No-GS hybrid ensemble at wells
    full_at_wells_gs : np.ndarray (n_real, n_wells)  GS hybrid ensemble at wells

    Returns
    -------
    dict with keys: grid, well  each containing No GS and GS metrics
    """

    #  Grid-level 
    Z_beta_gslib  = np.flipud(results_['Z_beta'])
    z_grid_flat   = Z_beta_gslib.ravel()
    var_true_grid = float(np.var(z_grid_flat))

    var_grid_A = float(np.mean([np.var(h) for h in results_["hybrid"]]))
    var_grid_B = float(np.mean([np.var(h) for h in results_["hybrid_GS"]]))

    p10_grid_A, p90_grid_A = np.percentile(results_["hybrid"],    [10, 90], axis=0)
    p10_grid_B, p90_grid_B = np.percentile(results_["hybrid_GS"], [10, 90], axis=0)

    coverage_grid_A = float(np.mean(
        (z_grid_flat >= p10_grid_A.ravel()) & (z_grid_flat <= p90_grid_A.ravel())
    ))
    coverage_grid_B = float(np.mean(
        (z_grid_flat >= p10_grid_B.ravel()) & (z_grid_flat <= p90_grid_B.ravel())
    ))

    resid_grid_A = Z_beta_gslib - results_["T_hat_grid"]
    resid_grid_B = Z_beta_gslib - results_["T_hat_grid_gs"]

    cov_grid_A = scov(results_['T_hat_grid'],    resid_grid_A)
    cov_grid_B = scov(results_['T_hat_grid_gs'], resid_grid_B)

    pct_cov_grid_A = cov_grid_A / var_true_grid * 100 if var_true_grid > 1e-12 else 0.0
    pct_cov_grid_B = cov_grid_B / var_true_grid * 100 if var_true_grid > 1e-12 else 0.0

    vif_grid_A = var_grid_A / var_true_grid
    vif_grid_B = var_grid_B / var_true_grid

    # Goodness scores (Deutsch 1997)  uses compute_coverage_curve / deutsch_goodness
    # from cell 24; hybrid arrays are already in GSLIB convention, matching Z_beta_gslib
    nom_A, act_A   = compute_coverage_curve(Z_beta_gslib, np.array(results_["hybrid"]))
    goodness_grid_A = deutsch_goodness(nom_A, act_A)
    nom_B, act_B   = compute_coverage_curve(Z_beta_gslib, np.array(results_["hybrid_GS"]))
    goodness_grid_B = deutsch_goodness(nom_B, act_B)

    # Sill ratio: var(estimated residuals at wells) / var(true residuals at wells)
    # Using sample variance rather than raw optimizer sill because with itrans=1
    # in SGS the simulation variance is governed by the well-level residual spread,
    # not the raw fitted sill (which can be degenerate).  Values near 1.0 are ideal;
    # <1 means the GP over-smoothed the residuals, >1 means it added variance.
    eps_sill = 1e-12
    var_true_resid = float(np.var(sampled_df["Res_Por"]))
    sill_ratio_A = float(np.var(sampled_df["Estimated_Residual"])) / (var_true_resid + eps_sill)
    sill_ratio_B = float(np.var(sampled_df["GS_Residual"]))        / (var_true_resid + eps_sill)

    # E-type mean error (bias) and RMSE  grid level, GSLIB convention
    e_type_grid_A = np.array(results_["hybrid"]).mean(axis=0)
    e_type_grid_B = np.array(results_["hybrid_GS"]).mean(axis=0)
    bias_grid_A  = float(np.mean(e_type_grid_A - Z_beta_gslib))
    bias_grid_B  = float(np.mean(e_type_grid_B - Z_beta_gslib))
    rmse_grid_A  = float(np.sqrt(np.mean((e_type_grid_A - Z_beta_gslib) ** 2)))
    rmse_grid_B  = float(np.sqrt(np.mean((e_type_grid_B - Z_beta_gslib) ** 2)))
    rmse_ratio   = rmse_grid_A / (rmse_grid_B + eps_sill)  # >1 means GS improved accuracy

    # Projection coefficient alpha (GS correction strength)  recomputed from well data
    # alpha = <demean(trend), demean(residual)> / <demean(residual), demean(residual)>
    # (matches gram_schmidt_orthogonalization internals; A=residual, B=trend)
    _a = sampled_df["Estimated_Residual"].values - sampled_df["Estimated_Residual"].mean()
    _b = sampled_df["Estimated_Trend"].values    - sampled_df["Estimated_Trend"].mean()
    alpha_gs = float(np.dot(_b, _a) / (np.dot(_a, _a) + eps_sill))

    #  Sample Location-level 
    obs           = sampled_df['Porosity'].values
    var_true_well = float(np.var(obs))

    var_well_A = float(np.mean(np.var(full_at_wells,    axis=1)))
    var_well_B = float(np.mean(np.var(full_at_wells_gs, axis=1)))

    cov_well_A = scov(sampled_df["Estimated_Trend"], sampled_df["Estimated_Residual"])
    cov_well_B = scov(sampled_df["GS_Trend"],        sampled_df["GS_Residual"])

    pct_cov_well_A = cov_well_A / var_true_well * 100 if var_true_well > 1e-12 else 0.0
    pct_cov_well_B = cov_well_B / var_true_well * 100 if var_true_well > 1e-12 else 0.0

    vif_well_A = var_well_A / var_true_well
    vif_well_B = var_well_B / var_true_well

    p10_well_A, p90_well_A = np.percentile(full_at_wells,    [10, 90], axis=0)
    p10_well_B, p90_well_B = np.percentile(full_at_wells_gs, [10, 90], axis=0)
    coverage_well_A = float(np.mean((obs >= p10_well_A) & (obs <= p90_well_A)))
    coverage_well_B = float(np.mean((obs >= p10_well_B) & (obs <= p90_well_B)))

    #  Return 
    return {
        "grid": {
            "no_gs": {
                "coverage":   coverage_grid_A,
                "variance":   var_grid_A,
                "covariance": cov_grid_A,
                "pct_cov":    pct_cov_grid_A,
                "vif":        vif_grid_A,
                "goodness":   goodness_grid_A,
                "sill_ratio": sill_ratio_A,
                "bias":       bias_grid_A,
                "rmse":       rmse_grid_A,
            },
            "gs": {
                "coverage":   coverage_grid_B,
                "variance":   var_grid_B,
                "covariance": cov_grid_B,
                "pct_cov":    pct_cov_grid_B,
                "vif":        vif_grid_B,
                "goodness":   goodness_grid_B,
                "sill_ratio": sill_ratio_B,
                "bias":       bias_grid_B,
                "rmse":       rmse_grid_B,
            },
            "rmse_ratio":   rmse_ratio,
            "alpha_gs":     alpha_gs,
        },
        "well": {
            "no_gs": {
                "coverage":   coverage_well_A,
                "variance":   var_well_A,
                "covariance": cov_well_A,
                "pct_cov":    pct_cov_well_A,
                "vif":        vif_well_A,
            },
            "gs": {
                "coverage":   coverage_well_B,
                "variance":   var_well_B,
                "covariance": cov_well_B,
                "pct_cov":    pct_cov_well_B,
                "vif":        vif_well_B,
            },
        },
    }


def flatten_metrics(metrics, dataset_name):
    """Flatten the nested get_metrics() dict into one flat row for CSV logging."""
    return {
        'dataset_name':           dataset_name,
        # Grid level
        'grid_coverage_no_gs':    metrics['grid']['no_gs']['coverage'],
        'grid_coverage_gs':       metrics['grid']['gs']['coverage'],
        'grid_variance_no_gs':    metrics['grid']['no_gs']['variance'],
        'grid_variance_gs':       metrics['grid']['gs']['variance'],
        'grid_vif_no_gs':         metrics['grid']['no_gs']['vif'],
        'grid_vif_gs':            metrics['grid']['gs']['vif'],
        'grid_cov_no_gs':         metrics['grid']['no_gs']['covariance'],
        'grid_cov_gs':            metrics['grid']['gs']['covariance'],
        'grid_pct_cov_no_gs':     metrics['grid']['no_gs']['pct_cov'],
        'grid_pct_cov_gs':        metrics['grid']['gs']['pct_cov'],
        'grid_goodness_no_gs':    metrics['grid']['no_gs']['goodness'],
        'grid_goodness_gs':       metrics['grid']['gs']['goodness'],
        'grid_sill_ratio_no_gs':  metrics['grid']['no_gs']['sill_ratio'],
        'grid_sill_ratio_gs':     metrics['grid']['gs']['sill_ratio'],
        'grid_bias_no_gs':        metrics['grid']['no_gs']['bias'],
        'grid_bias_gs':           metrics['grid']['gs']['bias'],
        'grid_rmse_no_gs':        metrics['grid']['no_gs']['rmse'],
        'grid_rmse_gs':           metrics['grid']['gs']['rmse'],
        'grid_rmse_ratio':        metrics['grid']['rmse_ratio'],
        'grid_alpha_gs':          metrics['grid']['alpha_gs'],
        # Well level
        'well_coverage_no_gs':    metrics['well']['no_gs']['coverage'],
        'well_coverage_gs':       metrics['well']['gs']['coverage'],
        'well_variance_no_gs':    metrics['well']['no_gs']['variance'],
        'well_variance_gs':       metrics['well']['gs']['variance'],
        'well_vif_no_gs':         metrics['well']['no_gs']['vif'],
        'well_vif_gs':            metrics['well']['gs']['vif'],
        'well_cov_no_gs':         metrics['well']['no_gs']['covariance'],
        'well_cov_gs':            metrics['well']['gs']['covariance'],
        'well_pct_cov_no_gs':     metrics['well']['no_gs']['pct_cov'],
        'well_pct_cov_gs':        metrics['well']['gs']['pct_cov'],
    }

def get_well_grid_idx(sampled_df, dx, ny):
    ix = (sampled_df["X"].values / dx).astype(int).clip(0, nx - 1)
    iy = (sampled_df["Y"].values / dx).astype(int).clip(0, ny - 1)
    iy_gslib = (ny - 1) - iy   # flip to GSLIB convention
    return ix, iy_gslib


### **Experiment Runs**

In [ ]:
from plot_variogram_reproduction import *


def run_experiment_for_dataset(
    dataset_name, base_dir,
    n_sgs_realizations=10, n_gp_trials=100,
    nx=100, ny=100, dx=10.0, L=1000.0,
):
    """
    Run the full analysis pipeline for a single experiment folder and
    return a flat metrics dict ready for CSV logging.
    All plots are saved to experiments/{dataset_name}/ by their own functions.
    plt.close('all') is called at the end to free memory.
    """

    results_, results_df = run_single_experiment(
        base_dir, n_sgs_realizations=n_sgs_realizations, n_gp_trials=n_gp_trials, use_wt_col = True,
        nx=nx, ny=ny, dx=dx, L=L, dataset_name=dataset_name, verbose=False,
    )

    import unittest.mock as _mock
    with _mock.patch("matplotlib.pyplot.show"):
        plot_reproduction_check_scatter(results_, results_df, base_dir=base_dir, dataset_name=dataset_name, n_real=n_sgs_realizations)
    
        plot_reconstructed_fields(base_dir, results_, results_df, L, dataset_name, figsize=(18, 10))
    
        plot_variance_decomposition(
            point_trend_A    = results_df["Estimated_Trend"],
            point_residual_A = results_df["Estimated_Residual"],
            point_trend_B    = results_df["GS_Trend"],
            point_residual_B = results_df["GS_Residual"],
            Z_samples        = results_df["Porosity"],
            dataset_name=dataset_name, base_dir=base_dir,
        )
    
        plot_residual_variograms(sampled_df=results_df, dataset_name=dataset_name, base_dir=base_dir)
    
        plot_reproduction_check_hist_cdf(
            results_,
            results_df,
            n_sgs_realizations,
            data_col_no_gs="Estimated_Residual",
            data_col_gs="GS_Residual",
            weight_col="Wts_cell",
            sim_key_no_gs="sgs_A",
            sim_key_gs="sgs_B",
            bins=30,
            figsize=(14, 10),
            base_dir=base_dir,
            dataset_name=dataset_name,
            save_path=None,
        )
    
            
        prepared = prepare_summary_maps(results_, L)
        plot_gs_comparison(prepared, results_df, figsize=(18, 10), save_path=f"{base_dir}/{dataset_name}/workflow_comparison.png")
    
    plt.close('all')   # free memory all plots are saved to disk

    ix, iy_g = get_well_grid_idx(results_df, dx, ny)

    # Hybrid ensemble at wells: (n_real, n_wells)
    full_at_wells    = results_["sgs_A"][:, iy_g, ix] + results_df["Estimated_Trend"].values
    full_at_wells_gs = results_["sgs_B"][:, iy_g, ix] + results_df["GS_Trend"].values


    metrics = get_metrics(results_, results_df, L, full_at_wells, full_at_wells_gs)

    flattened_metrics = flatten_metrics(metrics, dataset_name)
    results_df.to_csv(os.path.join(base_dir, dataset_name, "sampled_wells_result.csv"), index=False)

    return flattened_metrics, results_df


In [ ]:
base_dir = "GS_experiment_datasets"

### Run Experiment Batch

In [ ]:
def run_gp_only_all_datasets(
    base_dir,
    n_gp_trials=50,
    nx=100, ny=100, dx=10.0, L=1000.0,
    save_csv=None,
):
    """
    Run GP regression + GS correction for every dataset folder in base_dir.
    Saves per-dataset artefacts:
      - sampled_wells_result.csv  (wells with all estimated columns)
      - T_hat_grid.npy / T_hat_grid_gs.npy / kriged_deltas.npy
      - reconstructed_fields.png
      - variance_decomposition.png
    No variogram fitting or SGS is performed.

    Columns in returned / saved summary CSV
    ----------------------------------------
    dataset_name, actual_trend_var_ratio, pct_var_trend, pct_var_residual,
    alpha_gs, pct_var_cov, effective_target
    """
    _required = ['metadata.json', 'Z_gaussian.npy', 'sampled_wells.csv']
    all_folders = sorted([
        d for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d))
        and all(os.path.exists(os.path.join(base_dir, d, f)) for f in _required)
    ])

    # Resume-safe: skip datasets whose row is already in the CSV
    if save_csv is not None and os.path.exists(save_csv):
        done = set(pd.read_csv(save_csv)['dataset_name'].tolist())
        print(f"Resuming: {len(done)} already done, {len(all_folders) - len(done)} remaining.")
    else:
        done = set()
    pending = [f for f in all_folders if f not in done]

    xs = np.linspace(dx / 2, L - dx / 2, nx)
    ys = np.linspace(dx / 2, L - dx / 2, ny)
    gx, gy = np.meshgrid(xs, ys)
    grid_coords = np.column_stack([gx.ravel(), gy.ravel()])

    rows = []
    eps  = 1e-12

    for dataset_name in tqdm(pending, desc="GP-only", unit="dataset"):
        try:
            dataset_dir = os.path.join(base_dir, dataset_name)

            with open(os.path.join(dataset_dir, 'metadata.json')) as f:
                meta = json.load(f)

            sampled_df = pd.read_csv(os.path.join(dataset_dir, 'sampled_wells.csv'))
            Z_beta     = np.load(os.path.join(dataset_dir, 'Z_gaussian.npy'))

            dataset_id       = meta['combo_index']
            actual_ratio     = meta['actual_trend_to_total_var']
            offset_gpr_range = meta['range_ratio'] * 1000.0 + 150

            #  Declustering 
            try:
                wts_cell, _, _ = geostats.declus(
                    sampled_df, 'X', 'Y', 'Porosity',
                    iminmax=1, noff=10, ncell=100, cmin=1, cmax=5000,
                )
            except IndexError:
                wts_cell = np.ones(len(sampled_df))
            sampled_df['Wts_cell'] = wts_cell
            sampled_df[5]          = wts_cell   # mirror integer-key column used elsewhere

            #  GP trend model 
            gp_result = run_gp_for_dataset(
                samples           = sampled_df[["X", "Y"]],
                Z_samples         = sampled_df["Porosity"].values,
                true_trend_values = sampled_df["Trend_Por"].values,
                grid_coords       = grid_coords,
                target_var_ratio  = actual_ratio,
                nx=nx, ny=ny,
                n_trials          = n_gp_trials,
                n_blocks_side     = 2,
                ls_max            = 900,
                dataset_id        = dataset_id,
                verbose           = False,
                residual_range    = offset_gpr_range,
            )

            sampled_df["Estimated_Trend"]    = gp_result["point_trend"]
            sampled_df["Estimated_Residual"] = gp_result["point_residual"]

            #  Gram-Schmidt correction 
            _, point_trend_gs = gram_schmidt_orthogonalization(
                gp_result["point_residual"], gp_result["point_trend"]
            )
            sampled_df["GS_Trend"]       = point_trend_gs
            sampled_df["GS_Residual"]    = sampled_df["Porosity"].values - point_trend_gs
            sampled_df["GS_adjustments"] = point_trend_gs - gp_result["point_trend"]

            #  Trend grids (GSLIB convention: row 0 = y_max) 
            diff_kmap     = smooth_krige_GS_adjustments(sampled_df, dx=dx, nx=nx, ny=ny)
            T_hat_grid    = np.flipud(gp_result["trend_grid"].reshape(ny, nx))
            T_hat_grid_gs = T_hat_grid + diff_kmap

            np.save(os.path.join(dataset_dir, "T_hat_grid.npy"),    T_hat_grid)
            np.save(os.path.join(dataset_dir, "T_hat_grid_gs.npy"), T_hat_grid_gs)
            np.save(os.path.join(dataset_dir, "kriged_deltas.npy"), diff_kmap)

            #  Plots 
            results_stub = {
                "Z_beta":        Z_beta,
                "T_hat_grid":    T_hat_grid,
                "T_hat_grid_gs": T_hat_grid_gs,
                "kriged_deltas": diff_kmap,
            }

            import unittest.mock as _mock
            with _mock.patch("matplotlib.pyplot.show"):
                plot_reconstructed_fields(
                    base_dir, results_stub, sampled_df, L, dataset_name, figsize=(18, 10)
                )
                plot_variance_decomposition(
                    point_trend_A    = sampled_df["Estimated_Trend"],
                    point_residual_A = sampled_df["Estimated_Residual"],
                    point_trend_B    = sampled_df["GS_Trend"],
                    point_residual_B = sampled_df["GS_Residual"],
                    Z_samples        = sampled_df["Porosity"],
                    dataset_name=dataset_name, base_dir=base_dir,
                )
            plt.close('all')

            #  Save wells CSV 
            sampled_df.to_csv(
                os.path.join(dataset_dir, "sampled_wells_result.csv"), index=False
            )

            #  Summary metrics 
            var_z            = float(np.var(sampled_df["Porosity"].values))
            point_trend      = gp_result["point_trend"]
            point_residual   = gp_result["point_residual"]
            pct_var_trend    = float(np.var(point_trend))    / (var_z + eps) * 100
            pct_var_residual = float(np.var(point_residual)) / (var_z + eps) * 100
            pct_var_cov      = gp_result["cov_attribution_pct"] * 100

            _r       = point_residual - point_residual.mean()
            _t       = point_trend    - point_trend.mean()
            alpha_gs = float(np.dot(_t, _r) / (np.dot(_r, _r) + eps))

            eps_sill = 1e-12
            var_true_resid = float(np.var(sampled_df["Res_Por"]))
            sill_ratio_A = float(np.var(sampled_df["Estimated_Residual"])) / (var_true_resid + eps_sill)
            sill_ratio_B = float(np.var(sampled_df["GS_Residual"]))        / (var_true_resid + eps_sill)
            row = {
                'dataset_name':           dataset_name,
                'actual_trend_var_ratio': actual_ratio,
                'pct_var_trend':          pct_var_trend,
                'pct_var_residual':       pct_var_residual,
                'alpha_gs':               alpha_gs,
                'pct_var_cov':            pct_var_cov,
                'effective_target':       gp_result['effective_target'],
                "sill_ratio_no_gs":       sill_ratio_A,
                "sill_ratio_gs":          sill_ratio_B,
                'status':                 'ok',
            }

        except Exception as e:
            tqdm.write(f"  ERROR - {dataset_name}: {e}")
            row = {'dataset_name': dataset_name, 'status': 'error', 'error': str(e)}

        rows.append(row)

        # Write one row immediately so CSV is always up to date
        if save_csv is not None:
            row_df    = pd.DataFrame([row])
            write_hdr = not os.path.exists(save_csv)
            row_df.to_csv(save_csv, mode='a', header=write_hdr, index=False)

    summary_df = pd.DataFrame(rows)
    n_ok  = sum(1 for r in rows if r.get('status') == 'ok')
    n_err = sum(1 for r in rows if r.get('status') == 'error')
    print(f"Done. {n_ok} succeeded, {n_err} failed.")
    return summary_df


### **Train GP trend Model on all datasets**

In [ ]:
### Fit GP across all datasets, save summary CSV, and generate per-dataset plots/arrays for later SGS step.

base_dir = "GS_experiment_datasets"
RESULTS_CSV = f"{base_dir}/experiment_results.csv"

summary_df = run_gp_only_all_datasets(
    base_dir,
    n_gp_trials=200,
    nx=100, ny=100, dx=10.0, L=1000.0,
    save_csv=f"{base_dir}/experiment_results.csv",
)

### **Run Sequential Guassian Simulation on Corrected and Uncorrected Residuals across all datasets**

In [ ]:
def run_variogram_sgs_all_datasets(
    base_dir,
    results_csv,
    n_sgs_realizations=100,
    nx=100, ny=100, dx=10.0, L=1000.0,
):
    """
    Stage 2: Variogram fitting + SGS for every dataset that has already been
    processed by run_gp_only_all_datasets (i.e. sampled_wells_result.csv exists).

    Loads from disk:
      - sampled_wells_result.csv  (all GP/GS columns already present)
      - Z_gaussian.npy, T_hat_grid.npy, T_hat_grid_gs.npy, metadata.json

    Saves per-dataset:
      - sgs_A.npy, sgs_B.npy, hybrid_A.npy, hybrid_B.npy
      - residual_variograms.png
      - repro_hist_cdf.png
      - reproduction_check_scatter.png
      - workflow_comparison.png

    Appends / creates results_csv with one flat metrics row per dataset.
    Resume-safe: datasets already in results_csv are skipped.
    """
    _required = [
        'metadata.json', 'Z_gaussian.npy',
        'T_hat_grid.npy', 'T_hat_grid_gs.npy',
        'sampled_wells_result.csv',
    ]
    all_folders = sorted([
        d for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d))
        and all(os.path.exists(os.path.join(base_dir, d, f)) for f in _required)
    ])

    if os.path.exists(results_csv):
        done = set(pd.read_csv(results_csv)['dataset_name'].tolist())
        print(f"Resuming: {len(done)} already done, {len(all_folders) - len(done)} remaining.")
    else:
        done = set()
    pending = [f for f in all_folders if f not in done]

    import unittest.mock as _mock

    for dataset_name in tqdm(pending, desc="Variogram+SGS", unit="dataset"):
        try:
            dataset_dir = os.path.join(base_dir, dataset_name)

            with open(os.path.join(dataset_dir, 'metadata.json')) as f:
                meta = json.load(f)

            sampled_df   = pd.read_csv(os.path.join(dataset_dir, 'sampled_wells_result.csv'))
            Z_beta       = np.load(os.path.join(dataset_dir, 'Z_gaussian.npy'))
            T_hat_grid   = np.load(os.path.join(dataset_dir, 'T_hat_grid.npy'))
            T_hat_grid_gs= np.load(os.path.join(dataset_dir, 'T_hat_grid_gs.npy'))
            diff_kmap    = np.load(os.path.join(dataset_dir, 'kriged_deltas.npy'))

            dataset_id     = meta['combo_index']
            fallback_range = meta['range_ratio'] * 1000.0 / 2

            # ── Variogram fitting and SGS ─────────────────────────────────────────────
            sgs_seed       = (dataset_id * 1000) + 45
            sim_result, sampled_df = geostats_simulation(
                sampled_df, T_hat_grid, T_hat_grid_gs,
                fallback_range=fallback_range, seed=sgs_seed,
                n_realizations=n_sgs_realizations,
                nx=nx, ny=ny, dx=dx,
            )

            sgs_A    = sim_result["sgs_pre"]
            sgs_B    = sim_result["sgs_post"]
            hybrid_A = sim_result["hybrid_pre"]
            hybrid_B = sim_result["hybrid_post"]
            vdiag_A  = sim_result["vdiag_pre"]
            vdiag_B  = sim_result["vdiag_post"]


            np.save(os.path.join(dataset_dir, "sgs_A.npy"),    sgs_A)
            np.save(os.path.join(dataset_dir, "sgs_B.npy"),    sgs_B)
            np.save(os.path.join(dataset_dir, "hybrid_A.npy"), hybrid_A)
            np.save(os.path.join(dataset_dir, "hybrid_B.npy"), hybrid_B)

            # ── Assemble results_ dict (mirrors run_single_experiment output) ─
            results_ = {
                "Z_beta":        Z_beta,
                "hybrid":        hybrid_A,
                "hybrid_GS":     hybrid_B,
                "T_hat_grid":    T_hat_grid,
                "T_hat_grid_gs": T_hat_grid_gs,
                "sgs_A":         sgs_A,
                "sgs_B":         sgs_B,
                "kriged_deltas": diff_kmap,
                "vdiag_no_GS":   vdiag_A,
                "vdiag_GS":      vdiag_B,
            }

            # ── Plots ─────────────────────────────────────────────────────────
            with _mock.patch("matplotlib.pyplot.show"):
                plot_residual_variograms(
                    sampled_df=sampled_df, dataset_name=dataset_name, base_dir=base_dir)

                plot_reproduction_check_hist_cdf(
                    results_, sampled_df, n_sgs_realizations,
                    data_col_no_gs="Estimated_Residual",
                    data_col_gs="GS_Residual",
                    weight_col="Wts_cell",
                    sim_key_no_gs="sgs_A",
                    sim_key_gs="sgs_B",
                    bins=30, figsize=(14, 10),
                    base_dir=base_dir, dataset_name=dataset_name, save_path=None,
                )

                plot_reproduction_check_scatter(
                    results_, sampled_df,
                    base_dir=base_dir, dataset_name=dataset_name,
                    n_real=n_sgs_realizations,
                )

                prepared = prepare_summary_maps(results_, L)
                plot_gs_comparison(
                    prepared, sampled_df, figsize=(18, 10),
                    save_path=os.path.join(dataset_dir, "workflow_comparison.png"),
                )

            plt.close('all')

            # ── Metrics ───────────────────────────────────────────────────────
            ix, iy_g = get_well_grid_idx(sampled_df, dx, ny)
            full_at_wells    = sgs_A[:, iy_g, ix] + sampled_df["Estimated_Trend"].values
            full_at_wells_gs = sgs_B[:, iy_g, ix] + sampled_df["GS_Trend"].values

            metrics          = get_metrics(results_, sampled_df, L, full_at_wells, full_at_wells_gs)
            flat             = flatten_metrics(metrics, dataset_name)
            flat['status']   = 'ok'
            flat['error']    = ''

        except Exception as e:
            tqdm.write(f"  ERROR — {dataset_name}: {e}")
            flat = {'dataset_name': dataset_name, 'status': 'error', 'error': str(e)}

        row_df    = pd.DataFrame([flat])
        write_hdr = not os.path.exists(results_csv)
        row_df.to_csv(results_csv, mode='a', header=write_hdr, index=False)

    print(f"All done. Results saved to: {results_csv}")
    return pd.read_csv(results_csv)


In [ ]:
forward_df = run_variogram_sgs_all_datasets(
    base_dir="GS_experiment_datasets",
    results_csv="GS_experiment_datasets/results.csv",
    n_sgs_realizations=100,
)


### **Analyze Experiment Results**

In [ ]:
merged = pd.read_csv(f"{base_dir}/results.csv")
merged.shape


In [ ]:
df = merged[merged.status == "ok"].copy()

def _scatter_ax(ax, x, y, xlabel, ylabel, title):
    """Scatter with 45-degree reference line, black dots."""
    lo = min(x.min(), y.min())
    hi = max(x.max(), y.max())
    pad = (hi - lo) * 0.05
    lims = (lo - pad, hi + pad)

    ax.scatter(x, y, c="black", s=18, linewidths=0, alpha=0.75)
    ax.plot(lims, lims, "--", color="gray", linewidth=1.0, label="1:1")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(xlabel, fontsize=12.5)
    ax.set_ylabel(ylabel, fontsize=12.5)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_aspect("equal")
    ax.grid(True, linewidth=0.4, alpha=0.5)


# ── Fig 1: Sill Ratio | VIF | Goodness (1 × 3) ───────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4.5))
# fig2.suptitle("Diagnostic Metrics: With GS vs Without GS", fontsize=13, fontweight="bold", y=1.01)

_scatter_ax(axes2[0],
    x=df["grid_sill_ratio_no_gs"],
    y=df["grid_sill_ratio_gs"],
    xlabel="Without GS", ylabel="With GS", title="Sill Ratio")

_scatter_ax(axes2[1],
    x=df["grid_vif_no_gs"],
    y=df["grid_vif_gs"],
    xlabel="Without GS", ylabel="With GS", title="Variance Inflation Factor (VIF)")

_scatter_ax(axes2[2],
    x=df["grid_goodness_no_gs"],
    y=df["grid_goodness_gs"],
    xlabel="Without GS", ylabel="With GS", title="Goodness Score")

fig2.tight_layout()
fig2.savefig(f"{base_dir}/fig_diagnostics_scatter.png", dpi=1200, bbox_inches="tight")
plt.show()
